In [20]:
import os

from aeon.datasets import load_classification
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import polars as pl

In [21]:
multivariate_equal_length = [
    "ArticularyWordRecognition",
    "AtrialFibrillation",
    "BasicMotions",
    "Cricket",
    "DuckDuckGeese",
    "EigenWorms",
    "Epilepsy",
    "EthanolConcentration",
    "ERing",
    "FaceDetection",
    "FingerMovements",
    "HandMovementDirection",
    "Handwriting",
    "Heartbeat",
    "Libras",
    "LSST",
    "MotorImagery",
    "NATOPS",
    "PenDigits",
    "PEMS-SF",
    "PhonemeSpectra",
    "RacketSports",
    "SelfRegulationSCP1",
    "SelfRegulationSCP2",
    "StandWalkJump",
    "UWaveGestureLibrary",
]

In [22]:
fewvariate_equal_length = [
    "AtrialFibrillation",
    "BasicMotions",
    "Cricket",
    "EigenWorms",
    "Epilepsy",
    "EthanolConcentration",
    "ERing",
    "Handwriting",
    "Libras",
    "LSST",
    "PenDigits",
    "RacketSports",
    "SelfRegulationSCP1",
    "SelfRegulationSCP2",
    "StandWalkJump",
    "UWaveGestureLibrary",
]


In [23]:
raw_data_dir = os.path.join(os.environ["DATA_DIR"], "raw")
processed_data_dir = os.path.join(os.environ["DATA_DIR"], "preprocessed")


def tsc2pl(tsc_name: str):
    X_train, y_train = load_classification(
        tsc_name, split="train", extract_path=raw_data_dir
    )
    X_test, y_test = load_classification(
        tsc_name, split="test", extract_path=raw_data_dir
    )

    X_train = X_train.transpose(0, 2, 1)
    X_test = X_test.transpose(0, 2, 1)
    L = X_train.shape[1]

    trainval_df = pl.DataFrame(
        {"target": y_train}
        | {f"x{i}": X_train[..., i].tolist() for i in range(X_train.shape[-1])}
    )

    train_df, val_df = train_test_split(trainval_df, test_size=0.2, random_state=42)

    train_df = train_df.with_columns(split=pl.lit("train"))
    val_df = val_df.with_columns(split=pl.lit("val"))

    test_df = pl.DataFrame(
        {"target": y_test}
        | {f"x{i}": X_test[..., i].tolist() for i in range(X_test.shape[-1])}
    ).with_columns(split=pl.lit("test"))

    df = pl.concat([train_df, val_df, test_df])
    target_dtype = pl.Boolean if df["target"].n_unique() == 2 else pl.Int32
    df = df.with_columns(
        target=pl.col("target").cast(pl.Categorical).to_physical().cast(target_dtype),
        time=pl.lit(list(range(L))),
    )

    return df

In [24]:
import re


def to_snake(s):
    # First split sequences of uppercase letters (like "USA")
    s = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1_\2", s)
    # Then insert underscores before uppercase letters
    s = re.sub(r"([a-z])([A-Z])", r"\1_\2", s)
    # Convert to lowercase
    return s.lower()


In [25]:
stats = []

for ds in tqdm(fewvariate_equal_length):
    df = tsc2pl(ds)
    df.write_parquet(os.path.join(processed_data_dir, to_snake(ds) + ".parquet"))
    nfeats = len(df.columns) - 2
    nsamps = len(df)
    seqlen = df["time"].list.len().first()
    nclasses = df["target"].n_unique()
    stats.append(
        {
            "name": ds,
            "nfeats": nfeats,
            "nsamps": nsamps,
            "nclasses": nclasses,
            "seqlen": seqlen,
        }
    )

  0%|          | 0/16 [00:00<?, ?it/s]

In [26]:
with pl.Config(tbl_rows=1000):
    display(
        pl.DataFrame(stats).filter(
            pl.col("nsamps") > 100,
            pl.col("seqlen") > 5000,
            pl.col("nclasses") < 10,
        )
    )

name,nfeats,nsamps,nclasses,seqlen
str,i64,i64,i64,i64
"""EigenWorms""",7,259,5,17984
